In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType, ArrayType, DateType
import sys
import os
from delta import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql.utils import AnalysisException
from delta.tables import *
import io
import json
from pyspark.sql.functions import col, year, month, dayofmonth, expr

In [0]:
def create_spark_session():
    return SparkSession \
        .builder \
        .appName("File Streaming Demo") \
        .master("local[3]") \
        .config("spark.databricks.delta.schema.autoMerge.enabled", "true")\
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .enableHiveSupport()\
        .getOrCreate()

In [0]:
def create_deltaTable_insert_update_rows(spark:SparkSession,columns:list, location:str,merge_condition:str,df:DataFrame):
    if (DeltaTable.isDeltaTable(spark, location)):
        print('tabela delta existente')
        deltaTable = DeltaTable.forPath(spark, location)
        deltaTable.alias('tgt') \
            .merge(
                df.alias('src'),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    else:
        print('tabela delta inexistente')    
        DeltaTable \
            .create(spark) \
            .addColumns(columns) \
            .location(location) \
            .execute()
        deltaTable = DeltaTable.forPath(spark, location)
        deltaTable.alias('tgt') \
            .merge(
                df.alias('src'),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()


In [0]:
location_bronze = '/FileStore/bronze/dados_degue/chuvas'


#### Leiura camada bronze em dataframe spark

In [0]:
location_bronze = '/FileStore/bronze/dados_degue/chuvas'
df_bronze = spark.read.format('delta').load(location_bronze)
df_bronze.display()

data,mm,uf
2015-09-02,-9999.0,PA
2015-09-02,-9999.0,PA
2015-09-02,-9999.0,PA
2015-09-02,-9999.0,PA
2015-09-02,-9999.0,PA
2015-09-02,-9999.0,PA
2015-09-02,-9999.0,PA
2015-09-02,-9999.0,PA
2015-09-02,-9999.0,PA
2015-09-02,-9999.0,PA


#### Transformações camada silver 

In [0]:
df_silver = (
    df_bronze
    .withColumnRenamed('mm', 'mm')
    .withColumnRenamed('uf', 'estado')
    .withColumnRenamed('data', 'data_medicao')
    .withColumn('ano', year('data_medicao'))
    .withColumn('mes', month('data_medicao'))
    .withColumn('dia', dayofmonth('data_medicao'))
    .filter(col('mm').isNotNull() & (col('mm') >= 0))
)

df_silver.display()

data_medicao,mm,estado,ano,mes,dia
2017-01-01,9.4,PR,2017,1,1
2017-01-01,0.2,PR,2017,1,1
2017-01-01,0.0,PR,2017,1,1
2017-01-01,0.0,PR,2017,1,1
2017-01-01,0.0,PR,2017,1,1
2017-01-01,0.0,PR,2017,1,1
2017-01-01,0.0,PR,2017,1,1
2017-01-01,0.0,PR,2017,1,1
2017-01-01,0.0,PR,2017,1,1
2017-01-01,0.0,PR,2017,1,1


In [0]:
path_silver_chuva= '/FileStore/silver/dados_degue/chuvas'

#### Processo de Merge/Update para camada Silver

In [0]:
path_silver_chuva = '/FileStore/silver/dados_degue/chuvas'
merge_condition = expr("tgt.data_medicao = src.data_medicao and tgt.estado = src.estado and tgt.ano = src.ano and tgt.mes = src.mes and tgt.dia = src.dia")

columns = [
    StructField('data_medicao', DateType(), True),
    StructField('mm', DoubleType(), True),
    StructField('estado', StringType(), True),
    StructField('ano', IntegerType(), True),
    StructField('mes', IntegerType(), True),
    StructField('dia', IntegerType(), True)
]

create_deltaTable_insert_update_rows(spark, columns, path_silver_chuva, merge_condition, df_silver)


tabela delta inexistente


In [0]:
#dbutils.fs.rm('/FileStore/silver/dados_degue/chuvas/', True)